# Hypothesis 2 Validation:
## Small Data Modelling & Evaluating
---

## Hypothesis
Increasing dataset size and diversity improves accuracy.

## Objectives
- Train the same CNN model from the full dataset, but with a smaller subset (10% of the dataset)
- Compare performance metrics to validate Hypothesis 2.

## Inputs
* inputs/datasets/animals/small-data/train/
* inputs/datasets/animals/image/validation/
* inputs/datasets/animals/image/test/
* Image shape embeddings: outputs/v1/image_shape.pkl

## Outputs
- Image distribution plots for small data training, validation, and testing.
- Image augmentation pipeline.
- ML learning curves for both runs.
- Comparison table (subset vs full dataset).
- Model evaluation stored in pickle file.
- Predictions on random test images.
  
---

### Import libraries
A full import list for Notebook 04

In [18]:
# Core Python & utilities
import os
import random
import joblib
import numpy as np
import pandas as pd
import shutil

# Plotting & Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.image import imread

# Machine Learning / Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Metrics
from sklearn.metrics import classification_report, confusion_matrix

# Style for plots
sns.set_style("white")


## Set working directory

In [19]:
cwd = os.getcwd()
os.chdir('/workspaces/Animal_detection_camera')
print("You set a new current directory")

work_dir = os.getcwd()
work_dir

You set a new current directory


'/workspaces/Animal_detection_camera'

## Set output directory

In [20]:
version = 'v1'
file_path = f'outputs/{version}/small-data'
os.makedirs(file_path, exist_ok=True)

print(f"Saving results to: {file_path}")

Saving results to: outputs/v1/small-data


---

# Number of images in train, test and validation data
---

## Define Small Subset Creation Function


In [21]:
def create_subset(original_dir, subset_dir, fraction=0.1):
    os.makedirs(subset_dir, exist_ok=True)
    for class_name in os.listdir(original_dir):
        class_path = os.path.join(original_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        subset_class_path = os.path.join(subset_dir, class_name)
        os.makedirs(subset_class_path, exist_ok=True)

        files = os.listdir(class_path)
        sample_size = max(1, int(len(files) * fraction))
        sampled_files = random.sample(files, sample_size)

        for f in sampled_files:
            shutil.copy(os.path.join(class_path, f), os.path.join(subset_class_path, f))


### Create 10% small-data subset

In [22]:
small_data_dir = "inputs/datasets/animals/small-data/train"
if not os.path.exists(small_data_dir):
    print("Creating small-data subset...")
    create_subset("inputs/datasets/animals/image/train", small_data_dir, fraction=0.1)
else:
    print("Small-data subset already exists.")

Creating small-data subset...


## Set input directories
Set train, validation and test paths

In [28]:
my_data_dir = "inputs/datasets/animals"
train_path = small_data_dir
val_path = "inputs/datasets/animals/image/validation"
test_path = "inputs/datasets/animals/image/test"

## Set Labels

In [24]:
labels = os.listdir("inputs/datasets/animals/image/train")
print(f"Project Labels: {labels}")

Project Labels: ['lemur', 'snake', 'elephant', 'frog', 'chimpanzee', 'chinchilla', 'flamingo', 'mongoose', 'ostrich', 'ferret', 'camel', 'bee', 'mole', 'penguin', 'leopard', 'hawk', 'hedgehog', 'walrus', 'falcon', 'grasshopper', 'beaver', 'antelope', 'giraffe', 'duck', 'lizard', 'crab', 'goose', 'gorilla', 'jaguar', 'sheep', 'lynx', 'butterfly', 'panda', 'goat', 'deer', 'peacock', 'dog', 'whale', 'kangaroo', 'seal', 'cheetah', 'cow', 'iguana', 'hippopotamus', 'fox', 'cat', 'donkey', 'raccoon', 'blackbird', 'buffalo', 'koala', 'crocodile', 'dolphin', 'hyena', 'porcupine', 'bear', 'squid', 'spider', 'eagle', 'bison', 'owl', 'otter', 'snail', 'wolf']


## Set Image Shape

In [25]:
image_shape = joblib.load(filename=f"outputs/{version}/image_shape.pkl")
image_shape = (128, 128, 3)
print("Using image shape:", image_shape)

Using image shape: (128, 128, 3)


## Data Generators 
Small vs Full Datasets

In [26]:
datagen = ImageDataGenerator(rescale=1./255)

train_small_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/small-data/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

train_full_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/validation",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)


Found 997 images belonging to 64 classes.
Found 10050 images belonging to 64 classes.
Found 1400 images belonging to 64 classes.


---

## Image Distribution Analysis
---

### Prepare Frequency Counter

In [27]:
data = {'Set': [], 'Label': [], 'Frequency': []}
folders = ['small-data/train', 'image/validation', 'image/test']

### Count Images Per Set Of Labels

In [29]:
for folder in folders:
    folder_path = os.path.join(my_data_dir, folder)
    set_name = folder.split('/')[-1]
    
    for label in labels:
        n = len(os.listdir(os.path.join(folder_path, label)))
        data['Set'].append(set_name)
        data['Label'].append(label)
        data['Frequency'].append(n)
        print(f"* {set_name} - {label}: {n} images")

* train - lemur: 15 images
* train - snake: 15 images
* train - elephant: 20 images
* train - frog: 21 images
* train - chimpanzee: 15 images
* train - chinchilla: 15 images
* train - flamingo: 15 images
* train - mongoose: 15 images
* train - ostrich: 15 images
* train - ferret: 15 images
* train - camel: 15 images
* train - bee: 15 images
* train - mole: 12 images
* train - penguin: 15 images
* train - leopard: 15 images
* train - hawk: 15 images
* train - hedgehog: 15 images
* train - walrus: 15 images
* train - falcon: 15 images
* train - grasshopper: 15 images
* train - beaver: 15 images
* train - antelope: 15 images
* train - giraffe: 15 images
* train - duck: 15 images
* train - lizard: 15 images
* train - crab: 15 images
* train - goose: 15 images
* train - gorilla: 15 images
* train - jaguar: 13 images
* train - sheep: 15 images
* train - lynx: 15 images
* train - butterfly: 15 images
* train - panda: 15 images
* train - goat: 15 images
* train - deer: 15 images
* train - peac

### Create Dataframe with Frequencies